In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/masks1/masks_train1

CNV  DRUSEN  NORMAL


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from tqdm import tqdm
import os


In [ ]:
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import os

# === Custom Dataset care filtrează fișierele corupte ===
class OCTDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_per_class=6000):
        self.transform = transform
        self.data = []
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        class_counts = {cls: 0 for cls in self.classes}

        print("Scanning folders and filtering valid images...")
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for file in os.listdir(cls_dir):
                if class_counts[cls] >= max_per_class:
                    break
                file_path = os.path.join(cls_dir, file)
                try:
                    img = Image.open(file_path)
                    img.verify()  # verifică dacă fișierul e imagine validă
                    self.data.append((file_path, self.class_to_idx[cls]))
                    class_counts[cls] += 1
                except (UnidentifiedImageError, IOError):
                    print(f"Ignoring invalid image: {file_path}")

        print("Valid images per class:")
        for cls in self.classes:
            print(f"{cls}: {class_counts[cls]} images")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# === Transformări standard pentru model ===
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# === Creăm dataset-ul ===
dataset_path = "/content/drive/MyDrive/masks1/masks_train1"
dataset = OCTDataset(dataset_path, transform=transform, max_per_class=6000)

# === Split train/val ===
val_split = 0.2
n_total = len(dataset)
n_val = int(n_total * val_split)
n_train = n_total - n_val

train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

# === DataLoader pentru training rapid ===
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")


Scanning folders and filtering valid images...
Ignoring invalid image: /content/drive/MyDrive/masks1/masks_train1/CNV/CNV-1016042-128_mask.png
Valid images per class:
CNV: 2999 images
DRUSEN: 3000 images
NORMAL: 3000 images
Total training images: 7200
Total validation images: 1799


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(model.last_channel, len(classes))  # output 3 clase
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Mixed Precision
scaler = torch.cuda.amp.GradScaler()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 145MB/s]
/tmp/ipython-input-3534513938.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
num_epochs = 30  # crește dacă vrei

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # --- Training ---
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Train Loss: {train_loss/total:.4f} | Train Acc: {correct/total:.4f}")

    # --- Validation ---
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Val Loss: {val_loss/total:.4f} | Val Acc: {correct/total:.4f}")


Epoch 1/30


  0%|          | 0/225 [00:00<?, ?it/s]/tmp/ipython-input-2265140306.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 225/225 [01:01<00:00,  3.67it/s]


Train Loss: 0.2134 | Train Acc: 0.9192


100%|██████████| 57/57 [00:13<00:00,  4.12it/s]


Val Loss: 0.2362 | Val Acc: 0.9144

Epoch 2/30


...

Val Loss: 0.2811 | Val Acc: 0.9422

Epoch 30/30


100%|██████████| 225/225 [00:57<00:00,  3.89it/s]


Train Loss: 0.0136 | Train Acc: 0.9961


100%|██████████| 57/57 [00:13<00:00,  4.10it/s]

Val Loss: 0.2802 | Val Acc: 0.9444


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/oct_modelvar1_1_mobilnetv2.pth")

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 3)  # 3 clase: CNV, DRUSEN, NORMAL
model = model.to(device)

# === Loss și optimizer ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# === AMP scaler pentru mixed precision ===
scaler = torch.cuda.amp.GradScaler()

# === Training loop ===
num_epochs = 30  # ajustează după nevoie

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed precision
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        loop.set_postfix(loss=running_loss/total, acc=100*correct/total)

    # === Validation per epoch ===
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1} -> Train Loss: {running_loss/total:.4f}, Train Acc: {100*correct/total:.2f}% | Val Loss: {val_loss/val_total:.4f}, Val Acc: {100*val_correct/val_total:.2f}%")

/tmp/ipython-input-67344569.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/30: 100%|██████████| 225/225 [00:56<00:00,  3.98it/s, acc=84.9, loss=0.363]


Epoch 1 -> Train Loss: 0.3635, Train Acc: 84.90% | Val Loss: 0.2844, Val Acc: 89.94%


Epoch 2/30: 100%|██████████| 225/225 [00:57<00:00,  3.92it/s, acc=93.8, loss=0.164]


Epoch 2 -> Train Loss: 0.1640, Train Acc: 93.83% | Val Loss: 0.2341, Val Acc: 92.00%


Epoch 3/30: 100%|██████████| 225/225 [00:57<00:00,  3.90it/s, acc=96.3, loss=0.106]


...

Epoch 28 -> Train Loss: 0.0262, Train Acc: 99.11% | Val Loss: 0.8094, Val Acc: 86.55%


Epoch 29/30: 100%|██████████| 225/225 [00:58<00:00,  3.83it/s, acc=99, loss=0.0256]


Epoch 29 -> Train Loss: 0.0256, Train Acc: 99.04% | Val Loss: 0.3245, Val Acc: 94.72%


Epoch 30/30: 100%|██████████| 225/225 [00:57<00:00,  3.90it/s, acc=99.7, loss=0.00776]


Epoch 30 -> Train Loss: 0.0078, Train Acc: 99.71% | Val Loss: 0.3204, Val Acc: 94.55%


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/oct_modelvar1_2_resnet18.pth")